## Laufzeiten der Sortieralgorithmen

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import random
import math 

def selection_sort(a):
    for i in range(len(a)-1):
        best = i
        best_val = a[i]
        for j in range(i+1,len(a)):
            if a[j] < best_val:
                best = j
                best_val = a[j]
        a[best],a[i] = a[i],a[best]
        
def bubble_sort(a):
    getauscht = True
    while getauscht:
        getauscht = False
        for i in range(len(a)-1):
            if a[i] > a[i+1]:
                a[i],a[i+1]=a[i+1],a[i]
                getauscht = True
        
def insertion_sort(a):
    for i in range(1,len(a)):
        x = a[i]
        j = i
        while j > 0 and a[j-1] > x:
            a[j] = a[j-1]
            j = j - 1
        a[j] = x
        
def quick_sort(a, unten = 0, oben = None):
    if oben == None: oben = len(a)-1
    i , j = unten, oben
    mitte = (unten + oben) // 2
    pivot = a[mitte]
    while i <= j:
        while a[i] < pivot: i+=1
        while a[j] > pivot: j-=1
        if i <= j:
            tmp = a[i]
            a[i] = a[j]
            a[j] = tmp
            i+=1
            j-=1
    if unten < j: quick_sort(a,unten,j)
    if i < oben: quick_sort(a,i,oben)
    
def merge(a, b):
    i = j = 0
    c=[]
    while i < len(a) and j < len(b):
        if a[i] < b[j]:
            c.append(a[i])
            i+=1
        else:
            c.append(b[j])
            j+=1
    c+=a[i:]+b[j:]
    return c

def merge_sort(a):
    if len(a) <= 1: return a
    halb = len(a)//2
    b = a[0:halb]
    c = a[halb:]
    return merge(merge_sort(b),merge_sort(c))

def sift(a, start , end):
    i = start
    x = a[start]
    j = 2 * i + 1
    if j < end and a[j+1] < a[j]:
        j+=1
    while j <= end and a[j] < x:
        a[i] = a[j]
        i = j
        j = j * 2 + 1
        if j < end and a[j+1] < a[j]:
            j+=1
    a[i] = x


def heap_sort(a):
    n = len(a)
    for k in range ((n-2)//2,-1,-1):
        sift(a,k,n-1)
    for k in range(n-1,0,-1):
        a[0],a[k]=a[k],a[0]
        sift(a,0,k-1)

#### Hier die Zeitmessung durchführen

Wenn möglich für jede Anzahl ein paar Messungen durchführen und dann den Mittelwert nehmen.

In [2]:
N = 1          # N = anzahl der Elemente in Tausend
a = [random.randint(-N*10000,N*10000) for _  in range(N*1000)] 

In [3]:
%%time
selection_sort(a)

CPU times: total: 15.6 ms
Wall time: 22.5 ms


In [4]:
# Zelle für die Mittelwertberechnung

#### Hier die Ergebnisse eintragen

In [13]:
anzahl1 =  [1, 2, 4, 8, 16]
zeit_sel = [0, 0, 0, 0, 0]               # Selection-Sort
zeit_bb =  [0, 0, 0, 0, 0]               # Bubble-Sort
zeit_ins = [0, 0, 0, 0, 0]               # Insertion-Sort

In [14]:
anzahl2 =  [500, 1000, 2000, 5000 ,10000]
zeit_qui = [0, 0, 0, 0, 0]              # Quick-Sort
zeit_mer = [0, 0, 0, 0, 0]              # Merge-Sort
zeit_hea = [0, 0, 0, 0, 0]              # Heap-Sort
zeit_py =  [0, 0, 0, 0, 0]               # in Python eingebauter Sort

#### Ergebnisse für die Algorithmen mit Laufzeit $O(n^2)$

Zur Berechnung der Koeffizienten der Laufzeitfunktion $f(x) = a \cdot x^2 + b \cdot x + c$ nutzen wir die numpy-Funktion *polyfit*.

In [ ]:
a_sel = np.polyfit(anzahl1, zeit_sel, 2)
a_bb = np.polyfit(anzahl1, zeit_bb, 2)
a_ins = np.polyfit(anzahl1, zeit_ins, 2)

def f_sel(x):
    return a_sel[0]*x*x + a_sel[1]*x + a_sel[2]

def f_bb(x):
    return a_bb[0]*x*x + a_bb[1]*x + a_bb[2]

def f_ins(x):
    return a_ins[0]*x*x + a_ins[1]*x + a_ins[2]

fig, ax = plt.subplots()
ax.scatter(anzahl1,zeit_sel,label='selectionSort')
ax.scatter(anzahl1,zeit_bb,label='bubbleSort')
ax.scatter(anzahl1,zeit_ins,label='insertionSort')
ax.set_title('Laufzeit Selection/Bubble/Insertion Sort')
ax.set_xlabel('Größe der Liste in Tausend')
ax.set_ylabel('Zeit in Sekunden');
ax.legend();
xs = np.linspace(1,30,100)
ax.plot(xs,f_sel(xs),label='selectionSort');
ax.plot(xs,f_bb(xs),label='bubbleSort');
ax.plot(xs,f_ins(xs),label='insertionSort');

#### Ergebnisse für die Algorithmen mit Laufzeit $O(n \cdot \log(n))$


Für die Laufzeit setzen wir folgende Funktionsgleichung an:
$f(x) = a \cdot x \cdot \log(x) + b \cdot x + c \cdot \log(x) + d$. Zur Bestimmung der Koeffizienten nutzen wir die 
scipy-Funktion *curve_fit*.

In [ ]:
from scipy.optimize import curve_fit
def func(x, a, b, c, d):
    return a * np.log(x) * x + b * x + c * np.log(x) + d

a_qui, _ = curve_fit(func, anzahl2, zeit_qui)
a_mer, _ = curve_fit(func, anzahl2, zeit_mer)
a_hea, _ = curve_fit(func, anzahl2, zeit_hea)
a_py, _ = curve_fit(func, anzahl2, zeit_py)

def f_qui(x):
    return a_qui[0]*x*np.log(x) + a_qui[1]*x + a_qui[2]*np.log(x) + a_qui[3]

def f_mer(x):
    return a_mer[0]*x*np.log(x) + a_mer[1]*x + a_mer[2]*np.log(x) + a_mer[3]

def f_hea(x):
    return a_hea[0]*x*np.log(x) + a_hea[1]*x + a_hea[2]*np.log(x) + a_hea[3]

def f_py(x):
    return a_py[0]*x*np.log(x) + a_py[1]*x + a_py[2]*np.log(x) + a_py[3]

fig, ax = plt.subplots()
ax.scatter(anzahl2,zeit_qui,label='quickSort')
ax.scatter(anzahl2,zeit_mer,label='mergeSort')
ax.scatter(anzahl2,zeit_hea,label='heapSort')
ax.scatter(anzahl2,zeit_py,label='pythonSort')
ax.set_title('Laufzeit MergeSort/QuickSort/HeapSort/PythonSort')
ax.set_xlabel('Größe der Liste in Tausend')
ax.set_ylabel('Zeit in Sekunden');
ax.legend();
xs = np.linspace(500,10000,100)
ax.plot(xs,f_qui(xs),label='quickSort');
ax.plot(xs,f_mer(xs),label='mergeSort');
ax.plot(xs,f_hea(xs),label='heapSort');
ax.plot(xs,f_py(xs),label='pythonSort');

#### Berechnungen

Wieviele Elemente (in Tausend) können wir (ungefähr) in einer Minute sortieren?

- InsertionSort: ?
- SelectionSort: ?
- BubbleSort: ?
- HeapSort: ?
- MergeSort: ?
- QuickSort: ?
- PythonSort: ?


Wielange dauert es (ungefähr), eine Milliarde Zahlen zu sortieren?

- BubbleSort: ?
- SelectionSort: ?
- InsertionSort: ?
- HeapSort: ?
- MergeSort: ?
- QuickSort: ?
- PythonSort: ?